In [1]:
import os
import pandas as pd
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

In [2]:
base_dir = Path(r"D:\diplom\dataset_detect")
csv_path = base_dir / "oocyte_detect.csv"

In [3]:
class OocyteDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_name = str(row["name"]) + ".png"
        label = int(row["oocyte"])

        img_path = self.img_dir / img_name
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

In [5]:
df = pd.read_csv(csv_path)

In [6]:
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["oocyte"], random_state=42)

train_data = OocyteDataset(train_df, base_dir, transform)
test_data = OocyteDataset(test_df, base_dir, transform)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=16, shuffle=False)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2) 

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [8]:
def train(model, loader):
    model.train()
    total_loss = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [9]:
def evaluate(model, loader):
    model.eval()
    preds = []
    trues = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            predicted = torch.argmax(outputs, dim=1).cpu().numpy()

            preds.extend(predicted)
            trues.extend(labels.numpy())

    print(classification_report(trues, preds, digits=4))

In [10]:
EPOCHS = 10

for epoch in range(EPOCHS):
    loss = train(model, train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {loss:.4f}")

print("\n===== TEST METRICS =====")
evaluate(model, test_loader)

Epoch 1/10, Loss: 0.2308
Epoch 2/10, Loss: 0.0576
Epoch 3/10, Loss: 0.0421
Epoch 4/10, Loss: 0.0220
Epoch 5/10, Loss: 0.0111
Epoch 6/10, Loss: 0.0085
Epoch 7/10, Loss: 0.0139
Epoch 8/10, Loss: 0.0234
Epoch 9/10, Loss: 0.0089
Epoch 10/10, Loss: 0.0046

===== TEST METRICS =====
              precision    recall  f1-score   support

           0     1.0000    0.8889    0.9412        27
           1     0.9748    1.0000    0.9872       116

    accuracy                         0.9790       143
   macro avg     0.9874    0.9444    0.9642       143
weighted avg     0.9795    0.9790    0.9785       143



In [12]:
save_path = r"D:\diplom\weights"
os.makedirs(save_path, exist_ok=True)

torch.save(model.state_dict(), os.path.join(save_path, "resnet18_detect.pth"))